In [1]:
import pandas as pd

# Cargar los datos
customers = pd.read_csv('dades/customers.csv')
sales_history = pd.read_csv('dades/sales_history.csv')
sales_forecast = pd.read_csv('dades/sales_forecast_input.csv')

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. LIMPIAR LOS NULOS (La solución al error)
# Rellenamos los huecos con la mediana antes de pasarlos al modelo
sales_history['marketing_budget'] = sales_history['marketing_budget'].fillna(sales_history['marketing_budget'].median())
sales_history['planned_discount'] = sales_history['planned_discount'].fillna(sales_history['planned_discount'].median())

# 2. CONVERTIR NÚMEROS EN CATEGORÍAS (MULTICLASE)
sales_history['sales_class'] = pd.qcut(sales_history['sales_units'], q=3, labels=['Bajas', 'Medias', 'Altas'])

# 3. ELEGIR LAS VARIABLES DE TU MATRIZ
X = sales_history[['stock_units', 'list_price', 'marketing_budget', 'planned_discount']]
y = sales_history['sales_class'] 

# 4. SEPARAR LOS DATOS (80% para entrenar, 20% para probar)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. CREAR Y ENTRENAR EL MODELO MULTICLASE
modelo_logistico = LogisticRegression(solver='lbfgs', max_iter=1000)
modelo_logistico.fit(X_train, y_train)

# 6. HACER PREDICCIONES Y VER CÓMO DE BIEN LO HA HECHO
predicciones = modelo_logistico.predict(X_test)

print("--- REPORTE DEL MODELO LOGÍSTICO MULTICLASE ---")
print(classification_report(y_test, predicciones))

--- REPORTE DEL MODELO LOGÍSTICO MULTICLASE ---
              precision    recall  f1-score   support

       Altas       0.89      0.89      0.89        80
       Bajas       0.84      0.80      0.82        84
      Medias       0.67      0.70      0.68        74

    accuracy                           0.80       238
   macro avg       0.80      0.80      0.80       238
weighted avg       0.80      0.80      0.80       238



El modelo de clasificación múltiple demuestra que el stock inicial y el precio son fuertes indicadores de las ventas. El sistema resulta altamente fiable (89%) para identificar proactivamente los periodos de alta demanda, lo que permitiría a BiciTierra Market asegurar su inventario. Sin embargo, su menor precisión (67%) en la zona de ventas medias sugiere que clasificar por rangos genera ambigüedad. Para la decisión final, será más útil emplear un modelo de Regresión (que prediga la cifra exacta de unidades) en lugar de uno de Clasificación

In [3]:
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 0. ASEGURAR QUE LAS PARTICIONES EXISTEN (Solución al error)
sales_history['year'] = pd.to_datetime(sales_history['month']).dt.year
train = sales_history[(sales_history['year'] >= 2022) & (sales_history['year'] <= 2024)].copy()
val = sales_history[sales_history['year'] == 2025].copy()

# Limpiar nulos por si acaso
sales_history['marketing_budget'] = sales_history['marketing_budget'].fillna(sales_history['marketing_budget'].median())
sales_history['planned_discount'] = sales_history['planned_discount'].fillna(sales_history['planned_discount'].median())

# 1. Definir predictores (X) y el objetivo numérico continuo (y)
columnas_X = ['stock_units', 'list_price', 'marketing_budget', 'planned_discount']

X_train = train[columnas_X]
y_train = train['sales_units']  # Predicimos la columna numérica original

X_val = val[columnas_X]
y_val = val['sales_units']

# 2. Inicializar y entrenar el modelo Gradient Boosting
modelo_gb = GradientBoostingRegressor(random_state=42)
modelo_gb.fit(X_train, y_train)

# 3. Generar predicciones sobre los datos de validación
predicciones_gb = modelo_gb.predict(X_val)

# 4. Calcular métricas de evaluación
mae_gb = mean_absolute_error(y_val, predicciones_gb)
r2_gb = r2_score(y_val, predicciones_gb)

# Mostrar resultados
print("=== RESULTADOS GRADIENT BOOSTING EN VALIDACIÓN ===")
print(f"MAE (Error Medio Absoluto): {mae_gb:.2f} unidades")
print(f"R² (Coeficiente de Determinación): {r2_gb:.4f}")

=== RESULTADOS GRADIENT BOOSTING EN VALIDACIÓN ===
MAE (Error Medio Absoluto): 15.79 unidades
R² (Coeficiente de Determinación): 0.7889
